# Interactive Sample TOF Plot
This notebook mirrors the plotting logic from `plot_sample_tof.py` and adds a single control to choose the input data path.

In [15]:
from pathlib import Path
from typing import Literal

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import yaml
from IPython.display import clear_output, display

from joint_tof_opt.noise_calc import AdditiveGaussianToFModifier
from joint_tof_opt.plotting import load_plot_config
from joint_tof_opt.tof_batch_process import compute_tof_data_series


def load_gen_config():
    config_path = Path("../experiments/tof_config.yaml")
    if config_path.exists():
        with open(config_path) as f:
            return yaml.safe_load(f)
    raise FileNotFoundError("Could not find experiments/tof_config.yaml")


def plot_sample_tof(
    ppath: Path,
    plot_type: Literal["distribution", "density"],
    show_10pct_max_line: bool = False,
    noise_var: float = 0.1,
    x_max_ns: float | None = None,
):
    """
    Plot a sample time-of-flight histogram.

    Parameters:
    -----------
    ppath : str or Path
        Path to the .npz to the partial path table
    plot_type : str
        Type of plot: 'distribution' or 'density'
    show_10pct_max_line : bool
        If True, draw a horizontal line at 10% of the histogram maximum.
    x_max_ns : float | None
        Upper x-axis limit in nanoseconds. If None, keep default autoscaling.
    """
    gen_config = load_gen_config()
    gen_config["datapoint_count"] = 10  # Only generate 10 ToFs for quick loading
    gen_config["bin_count"] = 100  # More bins = more detail in the histogram
    # Load data
    tof_data = compute_tof_data_series(ppath, gen_config, False, False)
    modifier = AdditiveGaussianToFModifier(noise_var)
    modified_tof = modifier.modify(tof_data)

    modified_diff = (modified_tof.tof_series - tof_data.tof_series).sum().item()
    print(f"Modified ToF - Original ToF (first row): {modified_diff}")  # Print first 5 values of the first row

    # Get first row (first time point)
    tof_histogram = tof_data.tof_series.numpy()
    modified_tof_histogram = modified_tof.tof_series.numpy()
    tof_histogram = tof_histogram[0, :]
    modified_tof_histogram = modified_tof_histogram[0, :]

    # Convert from seconds to nanoseconds
    bin_edges_ns = tof_data.bin_edges.numpy() * 1e9

    # Calculate bin centers for the line plot
    bin_centers = (bin_edges_ns[:-1] + bin_edges_ns[1:]) / 2

    # Calculate bin widths
    bin_widths = np.diff(bin_edges_ns)

    # Normalize to density if requested
    if plot_type == "density":
        # Density: histogram / (sum * bin_width)
        y_values = tof_histogram / (np.sum(tof_histogram) * bin_widths)
        y_values_modified = modified_tof_histogram / (np.sum(modified_tof_histogram) * bin_widths)
        ylabel = "Probability Density"
    else:
        # Distribution: raw counts
        y_values = tof_histogram
        y_values_modified = modified_tof_histogram
        ylabel = "Count"

    # Load plot configuration
    load_plot_config()

    # Apply rcParams
    load_plot_config()
    # Get first color from the color cycle
    bar_color = plt.rcParams["axes.prop_cycle"].by_key()["color"][0]

    # Create figure
    fig, ax = plt.subplots(1, 2, figsize=(8, 3))
    ax[0].grid(True)
    ax[1].grid(True)
    ax[0].set_axisbelow(True)
    ax[1].set_axisbelow(True)

    # Plot bars with black edge
    ax[0].bar(
        bin_centers,
        y_values,
        width=bin_widths,
        color=bar_color,
        edgecolor="black",
        linewidth=0.5,
        label="Histogram",
    )
    ax[1].bar(
        bin_centers,
        y_values_modified,
        width=bin_widths,
        color=bar_color,
        edgecolor="black",
        linewidth=0.5,
        label="Noisy Histogram",
    )

    if show_10pct_max_line and np.size(y_values) > 0:
        max_value = float(np.max(y_values))
        if max_value > 0:
            ten_percent_max = 0.1 * max_value
            ax[0].axhline(
                y=ten_percent_max,
                color="crimson",
                linestyle="--",
                linewidth=1.2,
                label="10% of max",
            )

    if x_max_ns is not None:
        ax[0].set_xlim(0, x_max_ns)
        ax[1].set_xlim(0, x_max_ns)

    # Labels and title
    ax[0].set_xlabel("Time of Flight (ns)")
    ax[0].set_ylabel(ylabel)
    ax[0].set_title("Distribution Time-of-Flight (DTOF)")
    ax[1].set_xlabel("Time of Flight (ns)")
    ax[1].set_ylabel(ylabel)
    ax[1].set_title("Noisy Distribution Time-of-Flight (DTOF)")
    # ax.set_yscale("log")
    # ax.legend()

    plt.show()
    plt.close()


DATA_PATH_OPTIONS = [f"../data/experiment_{i:04d}.npz" for i in range(8)]

In [16]:
output = widgets.Output()


def run_interactive_plot(data_path: str, show_10pct_max_line: bool, x_max_ns: float, noise_var: float):
    with output:
        clear_output(wait=True)
        plot_sample_tof(
            Path(data_path),
            plot_type="density",
            show_10pct_max_line=show_10pct_max_line,
            x_max_ns=x_max_ns,
            noise_var=noise_var,
        )


data_selector = widgets.Dropdown(
    options=DATA_PATH_OPTIONS,
    value=DATA_PATH_OPTIONS[0],
    description="Data path:",
    layout=widgets.Layout(width="420px"),
)

show_10pct_line_checkbox = widgets.Checkbox(
    value=False,
    description="Show 10% of max line",
    indent=False,
)

x_max_slider = widgets.FloatSlider(
    value=4.0,
    min=1.0,
    max=4.0,
    step=0.1,
    description="X max (ns):",
    readout_format=".1f",
    continuous_update=False,
    layout=widgets.Layout(width="420px"),
)

noise_slider = widgets.FloatSlider(
    value=5000.0,
    min=0.0,
    max=10000.0,
    step=500.0,
    description="Noise variance:",
    readout_format=".2f",
    continuous_update=False,
    layout=widgets.Layout(width="420px"),
)

widgets.interactive_output(
    run_interactive_plot,
    {
        "data_path": data_selector,
        "show_10pct_max_line": show_10pct_line_checkbox,
        "x_max_ns": x_max_slider,
        "noise_var": noise_slider,
    },
)
display(widgets.VBox([data_selector, show_10pct_line_checkbox, x_max_slider, noise_slider, output]))